# 03_03 Support vector machines: why the book's SVM called everything ham

The book ended Chapter 3 by training a support vector classifier, finding that it labelled every message
ham, and concluding that logistic regression is better for this problem. That conclusion is wrong, and
finding out why teaches you what an SVM's kernel actually does.

**How this notebook works.** The same rhythm as every notebook in this course:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-03-teaching-a-machine-what-spam-looks-like", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'sentence_transformers': 'sentence-transformers',
           'nltk': 'nltk',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy',
           'matplotlib': 'matplotlib',
           'joblib': 'joblib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    import nltk
    for pkg in ['punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger_eng', 'maxent_ne_chunker_tab', 'words']:
        nltk.download(pkg, quiet=True)
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import time
import joblib
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.base import clone
from sklearn.exceptions import NotFittedError
from sklearn import metrics
from spamtools import load_sms, split
from nlpcheck import ask, guess, reveal, check_03_03

df = load_sms()
X_train, X_test, y_train, y_test = split(df)

## 1. Recall

**r6.** Lowering a spam filter's threshold does what? (a) recall up, precision down,
(b) recall down, precision up, (c) both up

**r7.** How did you stop training/serving skew? (a) by removing the cleaning,
(b) by cleaning the test set too, (c) by putting the cleaning inside the pipeline

In [ ]:
ask("r6", "")
ask("r7", "")

## 2. The book's SVM

`SVC` with an **RBF kernel**, which measures how alike two messages are as `exp(-gamma * distance²)`:
1 for identical messages, falling towards 0 as they differ. The book set `gamma='auto'`. It takes about
a second to train. Run it and see the book's result for yourself:

In [ ]:
t0 = time.time()
svc_auto = make_pipeline(TfidfVectorizer(), SVC(gamma="auto")).fit(X_train, y_train)
pred = svc_auto.predict(X_test)
print(f"trained in {time.time() - t0:.1f} s; accuracy {metrics.accuracy_score(y_test, pred):.3f};",
      (pred == "spam").sum(), "messages called spam")

None: the same 86.6 percent as the filter that flags nothing. To see why, look at the kernel itself.
`gamma='auto'` means `1 / number_of_features`, and TF-IDF made thousands of features. The next cell
prints the kernel's similarity between five different test messages. Predict the value between two
**different** messages, from 0 (nothing alike) to 1 (identical).

In [ ]:
vec = svc_auto[0]
n = len(vec.vocabulary_)
print(n, "features, so gamma =", 1 / n)
guess("kernel_similarity", None)   # a number from 0 to 1

In [ ]:
sample = vec.transform(X_test[:5])
K = rbf_kernel(sample, sample, gamma=1 / n)
print(np.round(K, 4))
reveal("kernel_similarity", round(float(K[0, 1]), 4))

Every pair of messages scores 0.9997: to this kernel, all messages look the same. TF-IDF vectors have
length 1, so the squared distance between two of them is at most 2, and `gamma` times 2 is almost nothing;
`exp` of almost nothing is almost 1. A kernel that cannot tell messages apart gives the SVM no way to separate
the classes, so it falls back to the majority. The model was not wrong for the problem; its setting made it
blind.

## 3. Your turn: an SVM that can see

Two fixes. `gamma='scale'`, scikit-learn's default since 2019, sets gamma from the spread of the data,
which for TF-IDF comes out near 1. Or skip the kernel: a **linear** SVM draws the widest straight boundary
in the TF-IDF space, which is what text usually needs, and trains faster. Compute the kernel at the scaled
gamma, then fit the linear SVM on the training messages (the line marked `# YOUR CODE HERE`).

In [ ]:
dense = vec.transform(X_train).toarray()
gamma_scale = 1 / (n * dense.var())
print("gamma='scale' is", round(gamma_scale, 3))
print(np.round(rbf_kernel(sample, sample, gamma=gamma_scale), 3))

svm = make_pipeline(TfidfVectorizer(), LinearSVC())
# YOUR CODE HERE: fit svm on the training messages and labels

try:
    pred = svm.predict(X_test)
    print(metrics.confusion_matrix(y_test, pred, labels=["ham", "spam"]))
    print(metrics.classification_report(y_test, pred, digits=3))
except NotFittedError:
    print("svm is not fitted yet: fill in the line marked YOUR CODE HERE and run this cell again.")

At the scaled gamma two unrelated messages score about 0.14 instead of 0.9997, and a pair that shares words
scores higher, so the kernel can tell messages apart and the RBF SVM works again. The linear SVM, once fitted, catches about 92 percent of the test spam at 99 percent
precision, **better than the logistic regression** at its default threshold. Compare the two side by side:

In [ ]:
from sklearn.linear_model import LogisticRegression
for name, m in [("logistic regression", make_pipeline(TfidfVectorizer(), LogisticRegression())),
                ("SVC, gamma='auto'", svc_auto),
                ("SVC, gamma='scale'", make_pipeline(TfidfVectorizer(), SVC())),
                ("LinearSVC", clone(svm))]:
    if m is not svc_auto:
        m.fit(X_train, y_train)
    p = m.predict(X_test)
    print(f"{name:20} precision {metrics.precision_score(y_test, p, pos_label='spam', zero_division=0):.3f}"
          f"  recall {metrics.recall_score(y_test, p, pos_label='spam'):.3f}"
          f"  F1 {metrics.f1_score(y_test, p, pos_label='spam'):.3f}")

## 4. Which features: exact words, or meaning?

Every model so far read TF-IDF, which knows exact strings and nothing else. A **sentence embedding** (page 01
of the chapter) knows meaning. In a pipeline the two are interchangeable: swap the first step and keep
everything else. `Embed()` is a scikit-learn step from `spamtools.py` that runs the embedding model in the
image on each message.

Embedding runs a neural network on every message, so this section uses 1,000 of the training messages and
600 of the test messages, both chosen with the same share of spam, instead of all 5,574. Their embeddings
were **prepared in the background when your session started** and saved under `out/.embeddings/`, so the
next cell normally takes seconds. If you got here within a few minutes of starting, the first run can take
up to about five minutes, because it computes whatever is not ready yet; later runs are quick. First, what
the two representations think of three messages:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from spamtools import Embed, embed

X_small, _, y_small, _ = train_test_split(X_train, y_train, train_size=1000, random_state=42, stratify=y_train)
X_check, _, y_check, _ = train_test_split(X_test, y_test, train_size=600, random_state=42, stratify=y_test)
t0 = time.time()
embed(X_small), embed(X_check)   # cached at session start; computed here if not ready yet
print(f"1,600 embeddings ready in {time.time() - t0:.0f} s")

three = ["You have won a free prize", "You have been selected for a complimentary reward",
         "Are you coming to dinner tonight?"]
tfidf_sim = cosine_similarity(TfidfVectorizer().fit(X_small).transform(three))
emb = embed(three)
print("TF-IDF similarity   prize vs reward", round(tfidf_sim[0, 1], 3), "| prize vs dinner", round(tfidf_sim[0, 2], 3))
print("embedding similarity prize vs reward", round(float(emb[0] @ emb[1]), 3), "| prize vs dinner", round(float(emb[0] @ emb[2]), 3))

The embedding sees that the prize and the reward mean the same thing, and TF-IDF barely does. Notice the
embedding scores the dinner invitation 0.77, not near 0: this model puts every English sentence in the
same broad region, so what matters is which texts are **nearer**, not the number itself.

Now the same classifier on each: logistic regression with `C=10`, which means weaker regularisation than the
default. On only 1,000 messages the default holds the weights so small that both models miss most of the
spam; that was measured, and it is why both get the same setting here.

Predict: which catches spam better on the 600 held-out messages, `"tfidf"` or `"embeddings"`?

In [ ]:
guess("features_winner", None)   # "tfidf" or "embeddings" 

In [ ]:
scores = {}
for name, first in [("tfidf", TfidfVectorizer()), ("embeddings", Embed())]:
    model = make_pipeline(first, LogisticRegression(C=10, max_iter=2000)).fit(X_small, y_small)
    p = model.predict(X_check)
    scores[name] = metrics.f1_score(y_check, p, pos_label="spam")
    print(f"{name:10}  precision {metrics.precision_score(y_check, p, pos_label='spam'):.3f}"
          f"  recall {metrics.recall_score(y_check, p, pos_label='spam'):.3f}  F1 {scores[name]:.3f}")
reveal("features_winner", max(scores, key=scores.get))

Embeddings, by four messages: both flag nothing that is not spam, and of the 80 spam messages in the check
set the embeddings catch 70 and TF-IDF 66. The extra four are spam written without the usual giveaway words,
where meaning is the only clue. It is a small margin, and it is not a law: scored on all 1,673 test messages,
with the same 1,000 training messages, the order flips, F1 0.908 for TF-IDF against 0.899. Spam mostly
gives itself away in exact strings (phone numbers, "txt", "claim"), which TF-IDF sees directly, so meaning
buys less here than you might expect. Where it pays most is when labelled examples are scarce: in Part 3,
describing the teams in one sentence each routes tickets with no labelled tickets at all. Choose the
features by measuring, the same way you chose the model.

## 5. Save and check

In [ ]:
joblib.dump(svm, "out/spam_svm.joblib")
check_03_03()

## 6. Exit ticket

**x3.** Which training points decide where an SVM's boundary goes? (a) all of them equally,
(b) the ones furthest from it, (c) the ones on the edge of the margin, the support vectors

In [ ]:
ask("x3", "")

Explain it back: the book concluded that logistic regression suits this problem better than an SVM. In two
sentences, what did its experiment actually show?

*Your explanation:* 